In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import precision_score, make_scorer

# Load the training and test data
train_data = pd.read_csv("/Users/eren/Desktop/412Project/bugs-train.csv")
test_data = pd.read_csv("/Users/eren/Desktop/412Project/bugs-test.csv")

# Filter out trivial entries if not already filtered
train_data = train_data[train_data['severity'] != 'trivial']

# Mapping severity levels to numerical values for training
severity_mapping = {
    'enhancement': 1,
    'minor': 2,
    'normal': 3,
    'major': 4,
    'blocker': 5,
    'critical': 6
}
train_data['severity'] = train_data['severity'].map(severity_mapping).astype(int)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(train_data['summary'], train_data['severity'], test_size=0.2, random_state=42, stratify=train_data['severity'])

# Initialize the TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
tfidf_features_train = tfidf_vectorizer.fit_transform(X_train)
tfidf_features_val = tfidf_vectorizer.transform(X_val)

# Define the Logistic Regression model with the best hyperparameters
best_model = LogisticRegression(C=1, penalty='l2', max_iter=1000, random_state=42)

# Train the best model on the training data
best_model.fit(tfidf_features_train, y_train)

# Evaluate the best model on the validation set
predicted_severity_val = best_model.predict(tfidf_features_val)
macro_precision_val = precision_score(y_val, predicted_severity_val, average='macro')
print(f"Macro Precision on Validation Set: {macro_precision_val}")

# Use the entire training set to train the best model
tfidf_features_full_train = tfidf_vectorizer.fit_transform(train_data['summary'])
best_model.fit(tfidf_features_full_train, train_data['severity'])

# Transform the test data with the same vectorizer
tfidf_features_test = tfidf_vectorizer.transform(test_data['summary'])

# Predict the severity for the test data
predicted_severity_test = best_model.predict(tfidf_features_test)

# Map numerical severity back to labels for submission
severity_labels = {v: k for k, v in severity_mapping.items()}
predicted_labels_test = [severity_labels[severity] for severity in predicted_severity_test]

# Prepare the submission DataFrame
submission_df = pd.DataFrame({
    'bug_id': test_data['bug_id'],
    'severity': predicted_labels_test
})

# Save the submission file
submission_file_path = "/Users/eren/Desktop/412Project/hadiya.csv"
submission_df.to_csv(submission_file_path, index=False)

# Print a message when the script completes
print("The model has been trained and predictions have been saved.")

Macro Precision on Validation Set: 0.7586102303224004
The model has been trained and predictions have been saved.
